In [ ]:
import pandas as pd
from collections import defaultdict

d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')

/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_63687/1368404073.py:4: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')


In [ ]:
df = d[['Smell_Word', 'year', 'Book']].copy()

df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df.dropna(subset=['Smell_Word', 'year', 'Book'])

df['source'] = df['Book'].astype(str).str.extract(r'^([a-zA-Z]+)')

df_gutenberg = df[df['source'].str.lower() == 'gu'].copy()

df_gutenberg = df_gutenberg[
    df_gutenberg['year'].between(1600, 1999)
]

print(df_gutenberg['source'].value_counts())

source
gu    994410
Name: count, dtype: int64


In [53]:
df_gutenberg = df[df['source'].str.lower() == 'gu'].copy()
print(f"tot: {len(df)}, Gutenberg: {len(df_gutenberg)}")

tot: 1535841, Gutenberg: 1002135


In [54]:
df_gutenberg = df_gutenberg[(df_gutenberg['year'] > 1600) & (df_gutenberg['year'] < 2000)]

word_freq_per_year_gu = defaultdict(lambda: defaultdict(int))
for _, row in df_gutenberg.iterrows():
    year = int(row['year'])
    words = str(row['Smell_Word']).lower().split('|')
    for word in words:
        word = word.strip()
        if year and word:
            word_freq_per_year_gu[year][word] += 1

records_gu = []
for year, freqs in word_freq_per_year_gu.items():
    for word, freq in freqs.items():
        records_gu.append({'word': word, 'frequency': freq, 'year': year})

df_word_freq_year_gu = pd.DataFrame(records_gu).sort_values(['year', 'frequency'], ascending=[True, False])

In [55]:
df_word_freq_year_gu = df_word_freq_year_gu[df_word_freq_year_gu['year'] < 2000]
df_word_freq_year_gu = df_word_freq_year_gu[df_word_freq_year_gu['year'] > 1600]
df_word_freq_year_gu

,word,frequency,year
45028,perfume,19,1603
45032,fmell,4,1603
45029,confume,2,1603
45030,contagious,2,1603
45031,perfumes,2,1603
...,...,...,...
46143,toothpaste,1,1999
46149,perfumes,1,1999
46150,essential,1,1999
46151,aromas,1,1999


In [57]:
from collections import defaultdict

def calculate_relative_frequency_per_word_df(df_gutenberg, categories):
    word_frequency = defaultdict(lambda: defaultdict(float))
    
    total_per_year = df_gutenberg.groupby('year')['frequency'].sum().to_dict()
    
    for _, row in df_gutenberg.iterrows():
        year = int(row['year'])
        word = row['word'].lower()
        freq = row['frequency']
        total = total_per_year[year]
        
        for category, category_words in categories.items():
            if word in category_words:
                word_frequency[word][year] += freq / total
    
    return word_frequency



def relative_frequency_per_word_df(word_frequency):
    for category, words in categories.items():
        print(f"--{category}--")
        for word in words:
            if word in word_frequency:
                print(f"Relative frequency for the word '{word}':")
                for year, frequency in sorted(word_frequency[word].items()):
                    print(f"    Year {year}: {frequency:.2%}")
                print()



categories = {  
    'stench/stinking': [
        'stink', 'stinch', 'stench', 'reek', 'whiff', 'fetor', 'foetor',
        'redolence', 'pong', 'niff', 'pungency', 'stinking', 'malodorous',
        'fetid', 'foetid', 'niffy', 'smelly', 'reeking', 'whiffy', 'pungent',
        'noisome', 'funky', 'musty', 'frowzy'
    ],
    
    'fragrance/fragrant': [
        'redolence', 'perfume', 'scent', 'aroma', 'fragrance', 'musk',
        'scented', 'aromatic', 'fragrant', 'redolent', 'sweet', 'fragrancy',
        'odoriferousness'
    ],
    
    'lacking_odour': [
        'odourless', 'odorless', 'scentless', 'unscented', 'deodorized',
        'deodorization', 'deodorizer', 'deodorant', 'unsmelling',
        'savourless', 'inodorate'
    ]
}



relative_word_frequency_per_year = calculate_relative_frequency_per_word_df(
    df_word_freq_year_gu,
    categories
)

# relative_frequency_per_word_df(relative_word_frequency_per_year)

In [ ]:
def print_top_words_per_category_per_year_raw(data, categories, top_n=10):
    """
    data: dict -> word -> year -> absolute frequency (raw count)
    categories: dict -> category -> list of words
    """
    for category, category_words in categories.items():

        years = set()
        for word in category_words:
            if word in data:
                years.update(data[word].keys())
        years = sorted(years)

        for year in years:
            category_words_year = {
                word: data[word][year]
                for word in category_words
                if word in data and year in data[word]
            }
            
            if not category_words_year:
                continue
            
            
            sorted_words = sorted(
                category_words_year.items(),
                key=lambda x: x[1],
                reverse=True
            )[:top_n]
            
            print(
                f"\nTop {top_n} words in the category "
                f"'{category}' for the year {year}:"
            )
            
            for word, freq in sorted_words:
                print(f"    {word}: {freq:.2%}")


# print_top_words_per_category_per_year_raw(
#     relative_word_frequency_per_year,
#     categories,
#     top_n=10
# )

In [58]:
def compute_entropy_dominance(df_word_freq_year_gu, categories, min_occ=0, min_years=3):
    """
    Considers only the years from 1600 to 2000, inclusive.
    """
    results = []

    # Time filter
    df_word_freq_year_gu = df_word_freq_year_gu[
        df_word_freq_year_gu['year'].between(1600, 2000)
    ]

    for category, category_words in categories.items():
        sub = df_word_freq_year_gu[
            df_word_freq_year_gu['word'].isin(category_words)
        ]

        # Sum raw frequencies for each year
        total_per_year = sub.groupby('year')['frequency'].sum()

        valid_years = sorted(
            total_per_year[total_per_year >= min_occ].index
        )

        entropy_list, dominance_list = [], []

        for year in valid_years:
            freqs = (
                sub[sub['year'] == year]
                .groupby('word')['frequency']
                .sum()
                .values
            )

            freqs = freqs / freqs.sum()

            H = -np.sum(freqs * np.log2(freqs + 1e-10))

            entropy_list.append(H)
            dominance_list.append(freqs.max())

        if len(valid_years) >= min_years:
            rho_h, p_h = spearmanr(valid_years, entropy_list)
            rho_d, p_d = spearmanr(valid_years, dominance_list)
        else:
            rho_h = p_h = rho_d = p_d = np.nan

        results.append({
            'category': category,
            'N_years': len(valid_years),
            'year_min': valid_years[0] if valid_years else None,
            'year_max': valid_years[-1] if valid_years else None,
            'rho_entropy': rho_h,
            'rho_dominance': rho_d,
        })

    return pd.DataFrame(results)

In [59]:
df_word_freq_year_gu

,word,frequency,year
45028,perfume,19,1603
45032,fmell,4,1603
45029,confume,2,1603
45030,contagious,2,1603
45031,perfumes,2,1603
...,...,...,...
46143,toothpaste,1,1999
46149,perfumes,1,1999
46150,essential,1,1999
46151,aromas,1,1999


In [60]:
import numpy as np
from scipy.stats import spearmanr

In [62]:
compute_entropy_dominance(df_word_freq_year_gu, categorie, min_occ=0)

,category,N_years,year_min,year_max,rho_entropy,rho_dominance
0,stench/stinking,373,1605,1998,0.647751,-0.565376
1,fragrance/fragrant,388,1603,1999,0.411660,-0.185456
2,lacking_odour,182,1790,1994,0.093139,-0.151777
